## XGBOOST

In [ ]:
# Suppress warnings
warnings.filterwarnings('ignore')

# Define paths
data_path = os.path.join("..", "data", "train-test")
models_path = os.path.join("..", "models")

# Create directories if they don't exist
os.makedirs(data_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)

# Load preprocessed data
print("Loading preprocessed data...")
train_df = pd.read_csv(os.path.join(data_path, "train_set.csv"))
test_df = pd.read_csv(os.path.join(data_path, "test_set.csv"))

# Separate features and target
X_train = train_df.drop(columns=['HasDiabetes'])
y_train = train_df['HasDiabetes']
X_test = test_df.drop(columns=['HasDiabetes'])
y_test = test_df['HasDiabetes']

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Target distribution (train): {y_train.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")
print(f"Target distribution (test): {y_test.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")

# HYPERPARAMETER TUNING WITH OPTUNA
def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',  # Optimize directly for AUC
        'booster': 'gbtree',
        'tree_method': 'hist',
        'n_jobs': 16,
        'random_state': 42,
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
        # Optional: allow scale_pos_weight tuning for recall bias
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 2.0)
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)

        evals = [(dtrain, 'train'), (dval, 'val')]
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=params['n_estimators'],
            evals=evals,
            early_stopping_rounds=50,
            verbose_eval=False
        )

        y_pred_proba = model.predict(dval)
        score = roc_auc_score(y_val, y_pred_proba)
        scores.append(score)

    return np.mean(scores)

print("\nStarting hyperparameter optimization with Optuna...")
study = optuna.create_study(direction='maximize', study_name='xgboost_diabetes')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\nBest parameters found:")
best_params = study.best_params
for key, value in best_params.items():
    print(f"  {key}: {value}")
print(f"Best CV AUC: {study.best_value:.4f}")

# TRAIN FINAL MODEL
print("\nTraining final XGBoost model with best parameters...")

# Add fixed params
best_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'booster': 'gbtree',
    'tree_method': 'hist',
    'n_jobs': 16,
    'random_state': 42,
    'verbosity': 0
})

# Create a held-out calibration set
X_train_split, X_calib, y_train_split, y_calib = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

# Final train/val split for early stopping
X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
    X_train_split, y_train_split, test_size=0.2, stratify=y_train_split, random_state=42
)

dtrain = xgb.DMatrix(X_train_final, label=y_train_final)
dval = xgb.DMatrix(X_val_final, label=y_val_final)

evals = [(dtrain, 'train'), (dval, 'val')]
final_model_native = xgb.train(
    best_params,
    dtrain,
    num_boost_round=best_params['n_estimators'],
    evals=evals,
    early_stopping_rounds=100,
    verbose_eval=False
)

# Wrapper for sklearn compatibility
class XGBWrapper:
    def __init__(self, booster):
        self.booster = booster

    def predict(self, X):
        dmatrix = xgb.DMatrix(X)
        return (self.booster.predict(dmatrix) > 0.5).astype(int)

    def predict_proba(self, X):
        dmatrix = xgb.DMatrix(X)
        proba = self.booster.predict(dmatrix)
        return np.column_stack([1 - proba, proba])

    @property
    def feature_importances_(self):
        return self.booster.get_score(importance_type='gain')

final_model = XGBWrapper(final_model_native)
print("Final model trained.")

# CALIBRATE ON HELD-OUT SET
print("\nCalibrating model on held-out calibration set...")
base_calib_model = xgb.XGBClassifier(
    **{k: v for k, v in best_params.items() if k not in ['eval_metric', 'verbosity']}
)
calibrated_model = CalibratedClassifierCV(base_calib_model, method='isotonic', cv='prefit')
calibrated_model.fit(X_calib, y_calib)

# Predictions
y_pred_proba = calibrated_model.predict_proba(X_test)[:, 1]
y_pred = calibrated_model.predict(X_test)

# EVALUATION
print("\nFinal Model Evaluation on Test Set:")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")

cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))

# THRESHOLD OPTIMIZATION
print("\nOptimizing decision threshold for clinical utility...")
precision_curve, recall_curve, thresholds_pr = precision_recall_curve(y_test, y_pred_proba)
# Avoid division by zero
f1_scores = 2 * (precision_curve * recall_curve) / (precision_curve + recall_curve + 1e-10)
optimal_idx = np.argmax(f1_scores[:-1])  # exclude last (undefined)
optimal_threshold = thresholds_pr[optimal_idx]

print(f"Optimal threshold for max F1: {optimal_threshold:.4f}")
print(f"Max F1 at this threshold: {f1_scores[optimal_idx]:.4f}")

y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)

# Correct specificity
cm_opt = confusion_matrix(y_test, y_pred_optimal)
specificity = cm_opt[0, 0] / cm_opt[0].sum()

print(f"\nPerformance at optimal threshold ({optimal_threshold:.4f}):")
print(f"Precision: {precision_score(y_test, y_pred_optimal):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_optimal):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_optimal):.4f}")
print(f"Specificity: {specificity:.4f}")

# VISUALIZATIONS
plt.figure(figsize=(12, 5))
# ROC
plt.subplot(1, 2, 1)
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.plot(fpr, tpr, label=f'XGBoost (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)

# PR Curve
plt.subplot(1, 2, 2)
plt.plot(recall_curve, precision_curve, label='Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Feature Importance
plt.figure(figsize=(10, 8))
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': [final_model.feature_importances_.get(f, 0) for f in X_train.columns]
}).sort_values('importance', ascending=False)

sns.barplot(data=feature_importance.head(15), x='importance', y='feature')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

# SHAP EXPLAINABILITY
print("\nGenerating SHAP explanations...")
explainer = shap.TreeExplainer(final_model_native)
shap_values = explainer.shap_values(X_test)

# Summary bar plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("SHAP Feature Importance")
plt.tight_layout()
plt.show()

# Beeswarm plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Summary Plot")
plt.tight_layout()
plt.show()

# SAVE
print("\nSaving models and results...")
joblib.dump(final_model_native, os.path.join(models_path, "xgboost_diabetes_final.joblib"))
joblib.dump(calibrated_model, os.path.join(models_path, "xgboost_diabetes_calibrated.joblib"))
joblib.dump(study, os.path.join(models_path, "optuna_study_xgboost.joblib"))

results = {
    'best_params': best_params,
    'test_metrics': {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': auc,
        'specificity_at_optimal_threshold': float(specificity)
    },
    'optimal_threshold': float(optimal_threshold),
    'feature_importance': feature_importance.to_dict('records'),
    'timestamp': str(datetime.now())
}

with open(os.path.join(models_path, "model_results_xgboost.json"), 'w') as f:
    json.dump(results, f, indent=4, default=str)

print("Models and results saved successfully.")

# FINAL RECOMMENDATIONS
print("\n" + "="*60)
print("FINAL MODEL RECOMMENDATIONS FOR CLINICAL USE")
print("="*60)
print(f"1. Use calibrated model with threshold: {optimal_threshold:.4f}")
print(f"2. Expected performance at this threshold:")
print(f"   - Sensitivity (Recall): {recall_score(y_test, y_pred_optimal):.2%}")
print(f"   - Specificity: {specificity:.2%}")
print(f"   - Precision: {precision_score(y_test, y_pred_optimal):.2%}")
print("="*60)

## CatBoost

In [ ]:
import os
import warnings
import json
import numpy as np
import pandas as pd
import catboost as cb
import optuna
import joblib
from datetime import datetime
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Suppress warnings
warnings.filterwarnings('ignore')

# Define paths
data_path = os.path.join("..", "data", "train-test")
models_path = os.path.join("..", "models")

# Create directories if they don't exist
os.makedirs(data_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)

# Load preprocessed data
print("Loading preprocessed data...")
train_df = pd.read_csv(os.path.join(data_path, "train_set.csv"))
test_df = pd.read_csv(os.path.join(data_path, "test_set.csv"))

# Separate features and target
X_train = train_df.drop(columns=['HasDiabetes'])
y_train = train_df['HasDiabetes']
X_test = test_df.drop(columns=['HasDiabetes'])
y_test = test_df['HasDiabetes']

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Target distribution (train): {y_train.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")
print(f"Target distribution (test): {y_test.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")

# HYPERPARAMETER TUNING WITH OPTUNA
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 1000),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 2.0),
        'eval_metric': 'AUC',
        'loss_function': 'Logloss',
        'random_seed': 42,
        'verbose': False,
        'thread_count': 16
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = cb.CatBoostClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            early_stopping_rounds=50,
            verbose=False
        )

        y_pred_proba = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, y_pred_proba)
        scores.append(score)

    return np.mean(scores)

print("\nStarting hyperparameter optimization with Optuna...")
study = optuna.create_study(direction='maximize', study_name='catboost_diabetes')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\nBest parameters found:")
best_params = study.best_params
for key, value in best_params.items():
    print(f"  {key}: {value}")
print(f"Best CV AUC: {study.best_value:.4f}")

# TRAIN FINAL MODEL
print("\nTraining final CatBoost model with best parameters...")

# Add fixed params
best_params.update({
    'eval_metric': 'AUC',
    'loss_function': 'Logloss',
    'random_seed': 42,
    'verbose': False,
    'thread_count': 16
})

# Create a held-out calibration set (20% of original train)
X_train_split, X_calib, y_train_split, y_calib = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

# Final train/val split for early stopping (20% of X_train_split)
X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
    X_train_split, y_train_split, test_size=0.2, stratify=y_train_split, random_state=42
)

# Train on combined train+val (i.e., X_train_split)
X_full_train = pd.concat([X_train_final, X_val_final])
y_full_train = pd.concat([y_train_final, y_val_final])

final_model = cb.CatBoostClassifier(**best_params)
final_model.fit(
    X_full_train, y_full_train,
    eval_set=(X_val_final, y_val_final),
    early_stopping_rounds=100,
    verbose=False
)

print("Base model trained.")

# CALIBRATE ON HELD-OUT SET
print("\nCalibrating model for reliable probabilities...")
calibrated_model = CalibratedClassifierCV(final_model, method='isotonic', cv='prefit')
calibrated_model.fit(X_calib, y_calib)
print("Model calibrated.")

# PREDICTIONS
y_pred_proba = calibrated_model.predict_proba(X_test)[:, 1]
y_pred = calibrated_model.predict(X_test)

# BASIC EVALUATION
print("\nFinal Model Evaluation on Test Set:")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")

# SAVE MODELS AND RESULTS
print("\nSaving models and results...")
joblib.dump(final_model, os.path.join(models_path, "catboost_diabetes_final.joblib"))
joblib.dump(calibrated_model, os.path.join(models_path, "catboost_diabetes_calibrated.joblib"))
joblib.dump(study, os.path.join(models_path, "optuna_study_catboost.joblib"))

results = {
    'best_params': best_params,
    'test_metrics': {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': auc
    },
    'timestamp': str(datetime.now())
}

with open(os.path.join(models_path, "model_results_catboost.json"), 'w') as f:
    json.dump(results, f, indent=4, default=str)

print("Models and results saved successfully.")

## Extra Trees

In [ ]:
import os
import warnings
import json
import numpy as np
import pandas as pd
import optuna
import joblib
from datetime import datetime
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Suppress warnings
warnings.filterwarnings('ignore')

# Define paths
data_path = os.path.join("..", "data", "train-test")
models_path = os.path.join("..", "models")
os.makedirs(data_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)

# Load data
print("Loading preprocessed data...")
train_df = pd.read_csv(os.path.join(data_path, "train_set.csv"))
test_df = pd.read_csv(os.path.join(data_path, "test_set.csv"))

X_train = train_df.drop(columns=['HasDiabetes'])
y_train = train_df['HasDiabetes']
X_test = test_df.drop(columns=['HasDiabetes'])
y_test = test_df['HasDiabetes']

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Target distribution (train): {y_train.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")
print(f"Target distribution (test): {y_test.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")

# HYPERPARAMETER TUNING WITH OPTUNA
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'class_weight': 'balanced',
        'n_jobs': 16,
        'random_state': 42
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = ExtraTreesClassifier(**params)
        model.fit(X_tr, y_tr)
        y_pred_proba = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, y_pred_proba))

    return np.mean(scores)

print("\nStarting hyperparameter optimization with Optuna...")
study = optuna.create_study(direction='maximize', study_name='extratrees_diabetes')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\nBest parameters found:")
best_params = study.best_params
for key, value in best_params.items():
    print(f"  {key}: {value}")
print(f"Best CV AUC: {study.best_value:.4f}")

# TRAIN FINAL MODEL
print("\nTraining final Extra Trees model...")
best_params.update({
    'class_weight': 'balanced',
    'n_jobs': 16,
    'random_state': 42
})

final_model = ExtraTreesClassifier(**best_params)
final_model.fit(X_train, y_train)  # Train on full training set

# CALIBRATE MODEL
print("Calibrating model for reliable probabilities...")
calibrated_model = CalibratedClassifierCV(final_model, method='isotonic', cv=3)
calibrated_model.fit(X_train, y_train)

# PREDICTIONS
y_pred_proba = calibrated_model.predict_proba(X_test)[:, 1]
y_pred = calibrated_model.predict(X_test)

# BASIC EVALUATION
print("\nFinal Model Evaluation on Test Set:")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")

# SAVE
print("\nSaving models and results...")
joblib.dump(final_model, os.path.join(models_path, "extratrees_diabetes_final.joblib"))
joblib.dump(calibrated_model, os.path.join(models_path, "extratrees_diabetes_calibrated.joblib"))
joblib.dump(study, os.path.join(models_path, "optuna_study_extratrees.joblib"))

results = {
    'best_params': best_params,
    'test_metrics': {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': auc
    },
    'timestamp': str(datetime.now())
}

with open(os.path.join(models_path, "model_results_extratrees.json"), 'w') as f:
    json.dump(results, f, indent=4, default=str)

print("Models and results saved successfully.")

## Random Forest

In [1]:
import os
import warnings
import json
import numpy as np
import pandas as pd
import optuna
import joblib
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Suppress warnings
warnings.filterwarnings('ignore')

# Define paths
data_path = os.path.join("..", "data",  "train-test")
models_path = os.path.join("..", "models")
os.makedirs(data_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)

# Load data
print("Loading preprocessed data...")
train_df = pd.read_csv(os.path.join(data_path, "train_set.csv"))
test_df = pd.read_csv(os.path.join(data_path, "test_set.csv"))

X_train = train_df.drop(columns=['HasDiabetes'])
y_train = train_df['HasDiabetes']
X_test = test_df.drop(columns=['HasDiabetes'])
y_test = test_df['HasDiabetes']

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Target distribution (train): {y_train.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")
print(f"Target distribution (test): {y_test.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")

# HYPERPARAMETER TUNING WITH OPTUNA
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'class_weight': 'balanced',
        'n_jobs': 16,
        'random_state': 42
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = RandomForestClassifier(**params)
        model.fit(X_tr, y_tr)
        y_pred_proba = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, y_pred_proba))

    return np.mean(scores)

print("\nStarting hyperparameter optimization with Optuna...")
study = optuna.create_study(direction='maximize', study_name='randomforest_diabetes')
study.optimize(objective, n_trials=65, show_progress_bar=True)

print(f"\nBest parameters found:")
best_params = study.best_params
for key, value in best_params.items():
    print(f"  {key}: {value}")
print(f"Best CV AUC: {study.best_value:.4f}")

# TRAIN FINAL MODEL ON FULL TRAINING SET
print("\nTraining final Random Forest model on full training set...")
best_params.update({
    'class_weight': 'balanced',
    'n_jobs': 16,
    'random_state': 42
})

final_model = RandomForestClassifier(**best_params)
final_model.fit(X_train, y_train)  #  Use full training data

# CALIBRATE MODEL
print("Calibrating model for reliable probabilities...")
calibrated_model = CalibratedClassifierCV(final_model, method='isotonic', cv=3)
calibrated_model.fit(X_train, y_train)

# PREDICTIONS
y_pred_proba = calibrated_model.predict_proba(X_test)[:, 1]
y_pred = calibrated_model.predict(X_test)

# BASIC EVALUATION
print("\nFinal Model Evaluation on Test Set:")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")

# SAVE MODELS AND RESULTS
print("\nSaving models and results...")
joblib.dump(final_model, os.path.join(models_path, "randomforest_diabetes_final.joblib"))
joblib.dump(calibrated_model, os.path.join(models_path, "randomforest_diabetes_calibrated.joblib"))
joblib.dump(study, os.path.join(models_path, "optuna_study_randomforest.joblib"))

results = {
    'best_params': best_params,
    'test_metrics': {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': auc
    },
    'timestamp': str(datetime.now())
}

with open(os.path.join(models_path, "model_results_randomforest.json"), 'w') as f:
    json.dump(results, f, indent=4, default=str)

print("Models and results saved successfully.")

c:\Users\hasit\OneDrive\Documents\Projects\HealthIntel\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading preprocessed data...


[I 2025-09-27 18:31:16,471] A new study created in memory with name: randomforest_diabetes


Train set: (281406, 18)
Test set: (70352, 18)
Target distribution (train): HasDiabetes
1.0    50.21%
0.0    49.79%
Name: proportion, dtype: object
Target distribution (test): HasDiabetes
1.0    50.21%
0.0    49.79%
Name: proportion, dtype: object

Starting hyperparameter optimization with Optuna...


Best trial: 0. Best value: 0.836381:   2%|▏         | 1/65 [00:49<52:41, 49.39s/it]

...

[I 2025-09-27 19:35:59,035] Trial 62 finished with value: 0.840729689042375 and parameters: {'n_estimators': 999, 'max_depth': 15, 'min_samples_split': 9, 'min_samples_leaf': 7, 'max_features': 'log2'}. Best is trial 57 with value: 0.8408255598687617.


Best trial: 57. Best value: 0.840826:  98%|█████████▊| 64/65 [1:05:59<01:15, 75.84s/it]

[I 2025-09-27 19:37:15,512] Trial 63 finished with value: 0.8404853143568014 and parameters: {'n_estimators': 967, 'max_depth': 12, 'min_samples_split': 8, 'min_samples_leaf': 8, 'max_features': 'log2'}. Best is trial 57 with value: 0.8408255598687617.


Best trial: 57. Best value: 0.840826: 100%|██████████| 65/65 [1:07:16<00:00, 62.10s/it]


[I 2025-09-27 19:38:33,118] Trial 64 finished with value: 0.8408201423670437 and parameters: {'n_estimators': 922, 'max_depth': 14, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': 'log2'}. Best is trial 57 with value: 0.8408255598687617.

Best parameters found:
  n_estimators: 868
  max_depth: 14
  min_samples_split: 9
  min_samples_leaf: 7
  max_features: log2
Best CV AUC: 0.8408

Training final Random Forest model on full training set...
Calibrating model for reliable probabilities...

Final Model Evaluation on Test Set:
Accuracy:  0.7630
Precision: 0.7432
Recall:    0.8068
F1-Score:  0.7737
ROC AUC:   0.8402

Saving models and results...
Models and results saved successfully.
